# Phase 6 Research Evaluation Notebook: Sentence-BERT vs TF-IDF Semantic Matching

## Project Title: AI Career Intelligence System
**Objective:** Quantitative & Qualitative evaluation of Transformer-based Sentence-BERT (`sentence-transformers/all-MiniLM-L6-v2`) vs traditional TF-IDF baseline for Resume-Job Matching.

### Evaluation Metrics Framework:
1. **Cosine Similarity Range & Distribution Analysis**
2. **Inference Latency & Memory Footprint**
3. **Ranking Effectiveness (NDCG@K, MRR)** *(Requires Ground-Truth Labeled Relevances)*
4. **Exact Keyword Overlap vs. Contextual Semantic Generalization**

In [ ]:
import math
import time
import json
from typing import List, Dict, Tuple

# Sample Benchmark Dataset (Candidate NLP Profiles vs Job NLP Descriptions)
benchmark_candidates = [
    {
        "id": "cand_01",
        "title": "Senior AI / Deep Learning Developer",
        "skills": ["Python", "PyTorch", "Sentence-Transformers", "FastAPI", "Docker", "PostgreSQL"],
        "experience": "5 years of experience developing deep neural network models, LLM pipelines, and vector search systems."
    },
    {
        "id": "cand_02",
        "title": "Frontend React Developer",
        "skills": ["React", "TypeScript", "Tailwind CSS", "Redux", "HTML5", "CSS3"],
        "experience": "3 years building web application user interfaces with React and modern responsive CSS framework components."
    }
]

benchmark_jobs = [
    {
        "id": "job_01",
        "title": "Senior AI / Machine Learning Engineer",
        "skills": ["Python", "PyTorch", "Transformers", "NLP", "FastAPI", "Vector Databases"],
        "description": "Seeking AI engineer to build fine-tuned Transformer models and semantic vector search engines."
    },
    {
        "id": "job_02",
        "title": "Full-Stack Software Engineer",
        "skills": ["React", "TypeScript", "Node.js", "Express", "PostgreSQL"],
        "description": "Building full-stack web applications with React frontend and Node backend."
    }
]

print(f"Loaded {len(benchmark_candidates)} benchmark candidates and {len(benchmark_jobs)} job postings.")

## 1. Metric Calculations (Precision, Recall, F1, NDCG@K, MRR)
*Note: In this evaluation notebook, ground-truth ranking annotations can be loaded from external benchmark datasets (e.g. ResumeIT, RecSys-Job-Dataset). Below is the mathematical evaluation function.*

In [ ]:
def calculate_mrr(rankings: List[List[int]]) -> float:
    """Calculates Mean Reciprocal Rank across query resume evaluations."""
    rr_sum = 0.0
    for rel_list in rankings:
        for rank_idx, rel in enumerate(rel_list):
            if rel > 0:
                rr_sum += 1.0 / (rank_idx + 1)
                break
    return rr_sum / max(1, len(rankings))

def calculate_ndcg_at_k(rel_list: List[int], k: int) -> float:
    """Calculates Normalized Discounted Cumulative Gain at rank K."""
    dcg = 0.0
    for i in range(min(k, len(rel_list))):
        rel = rel_list[i]
        dcg += (2**rel - 1) / math.log2(i + 2)
    
    ideal_rel = sorted(rel_list, reverse=True)
    idcg = 0.0
    for i in range(min(k, len(ideal_rel))):
        rel = ideal_rel[i]
        idcg += (2**rel - 1) / math.log2(i + 2)
    
    return dcg / idcg if idcg > 0 else 0.0

print("Evaluation metric calculation utilities successfully initialized.")

## 2. Research Findings Summary
- **Sentence-BERT (`all-MiniLM-L6-v2`)**: Generates 384-dimensional dense vectors that capture semantic similarity (e.g., matching 'PyTorch' with 'Deep Neural Networks') even when exact text strings differ.
- **TF-IDF Baseline**: Provides fast, interpretable keyword overlap baseline but suffers from vocabulary mismatch and synonym blindness.